# 03. Mold_Temperature_3/4 ↔ LH/RH 캐비티 매칭 검증

회의(9/26) 6-1에서 "Mold Temp 3/4 = 캐비티 LH/RH 온도"로 해석하기로 했지만 검증은 안 된 상태. 이 노트북에서 확인합니다.

**왜 단순 상관으로는 안 되나**: 한 샷의 LH·RH 두 행은 공정값이 **완전히 같음** (T3·T4 포함). 그래서 행 단위로 `cavity`와 T3/T4의 상관을 구하면 항상 0이고, 아무것도 알 수 없음. 대신 **간접 검증** 3가지를 합니다.

| 검증 | "T3 = LH, T4 = RH"가 맞다면 | 아니라면 (예: 냉각수 입구/출구, 같은 회로의 두 지점) |
|---|---|---|
| ① T4 − T3 부호 | 좌우 대칭 금형이라 **샷·기간에 따라 부호가 바뀌기도** 해야 자연스러움 | **항상 한쪽이 높음** (출구가 입구보다 따뜻함), 온도 수준이 올라가면 차이도 커짐 |
| ② 캐비티별 불량 | RH만 불량인 샷에서 **T4(또는 T4−T3)가 정상 샷과 다름** | 차이 없음 |
| ③ 한쪽 캐비티만 기록된 샷 (unlabeled) | 기록된 쪽 온도만 값이 있거나 좌우 차이 패턴이 달라짐 | 패턴 없음 |

⚠️ 어느 결과도 매칭을 **증명**하지는 못함. 최종 확인은 주최 측 문의(센서 위치)로.

In [ ]:
LINKED_DIR = './linked'
FIG_DIR = './figures/mold_temp'
import os, numpy as np, pandas as pd, matplotlib.pyplot as plt, matplotlib as mpl
from scipy import stats
os.makedirs(FIG_DIR, exist_ok=True)
from matplotlib import font_manager
_av = {f.name for f in font_manager.fontManager.ttflist}
for _f in ['Malgun Gothic', 'AppleGothic', 'NanumGothic', 'Noto Sans CJK KR', 'Noto Sans CJK JP']:
    if _f in _av:
        mpl.rcParams['font.family'] = _f; break
mpl.rcParams.update({'axes.unicode_minus': False, 'axes.spines.top': False, 'axes.spines.right': False})
BLUE, ORANGE, RED, GRAY = '#2a78d6', '#eb6834', '#d03b3b', '#c3c2b7'
PRODUCTS = ['cn7', 'rg3']
T3, T4 = 'Mold_Temperature_3_raw', 'Mold_Temperature_4_raw'

L = pd.read_csv(os.path.join(LINKED_DIR, 'linked_labeled.csv'), parse_dates=['TimeStamp'])
L = L[L['in_secondary']].copy()
L['d'] = L[T4] - L[T3]
L['ts_date'] = L['TimeStamp'].dt.date

## 0. 전제 확인: 같은 샷의 LH·RH 행은 T3/T4 값이 같은가

In [ ]:
chk = L.groupby(['product', 'shot_id']).agg(n=('cavity', 'size'), t3=(T3, 'nunique'), t4=(T4, 'nunique'))
print('두 행이 있는 샷 중 T3 또는 T4가 다른 샷 수:', int(((chk['n'] == 2) & ((chk['t3'] > 1) | (chk['t4'] > 1))).sum()))
print('→ 0이면 행 단위로 cavity와 T3/T4를 비교하는 건 의미 없음 (같은 값에 LH·RH 라벨만 다름)')

## ① T4 − T3 부호와 크기 (labeled + unlabeled 7개월)

In [ ]:
U = {}
for p in PRODUCTS:
    u = pd.read_csv(os.path.join(LINKED_DIR, f'linked_unlabeled_{p}.csv'), parse_dates=['TimeStamp'])
    u['d'] = u[T4] - u[T3]
    u['zero'] = (u[T3] == 0) & (u[T4] == 0)     # 센서 미기록(0)으로 보이는 행
    U[p] = u

rows = []
for p in PRODUCTS:
    for name, df in [('labeled', L[L['product'] == p]), ('unlabeled (0 제외)', U[p][~U[p]['zero']])]:
        rows.append({'product': p, '데이터': name, '행 수': len(df),
                     'T3 범위': f'{df[T3].min():.1f} ~ {df[T3].max():.1f}',
                     'T4 범위': f'{df[T4].min():.1f} ~ {df[T4].max():.1f}',
                     'T4 > T3': f"{(df['d'] > 0).mean():.1%}", 'T4 = T3': f"{(df['d'] == 0).mean():.1%}",
                     'T4 < T3': f"{(df['d'] < 0).mean():.1%}", 'd 중앙값': round(df['d'].median(), 2)})
    rows.append({'product': p, '데이터': 'unlabeled 중 T3=T4=0 행', '행 수': int(U[p]['zero'].sum())})
display(pd.DataFrame(rows).set_index(['product', '데이터']).fillna(''))

In [ ]:
# 온도 수준 vs 차이: 냉각수 입·출구라면 온도가 높을수록 차이도 커지는 경향
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2), sharey=True)
for ax, p in zip(axes, PRODUCTS):
    u = U[p][~U[p]['zero']].drop_duplicates('TimeStamp')
    ax.scatter(u[T3], u['d'], s=4, alpha=0.25, color=GRAY, label='unlabeled (3~10월)')
    l = L[L['product'] == p].drop_duplicates('shot_id')
    ax.scatter(l[T3], l['d'], s=10, alpha=0.8, color=BLUE, label='labeled (10월)')
    ax.axhline(0, color='#52514e', lw=0.8, ls='--')
    ax.set_title(f'{p.upper()} · T3 수준 vs (T4 − T3)', loc='left')
    ax.set_xlabel('Mold_Temperature_3 (℃)'); ax.set_ylabel('T4 − T3 (℃)')
    r = u[[T3, 'd']].corr('spearman').iloc[0, 1]
    ax.text(0.02, 0.95, f'unlabeled Spearman ρ = {r:.2f}', transform=ax.transAxes, fontsize=8, va='top')
axes[0].legend(frameon=False, fontsize=8, loc='upper right')
fig.tight_layout(); fig.savefig(os.path.join(FIG_DIR, 'level_vs_diff.png'), dpi=150); plt.show()

## ② 캐비티별 불량 패턴 vs T3 / T4 / (T4 − T3)

샷을 `정상 / RH만 불량 / LH만 불량 / 양쪽 불량`으로 나누고 Cliff's δ(불량 쪽 값이 클수록 +)로 비교합니다.
날짜 효과를 빼기 위해 **일자 내**(날짜별 중앙값을 뺀 값) 결과도 같이 봅니다.

In [ ]:
piv = L.pivot_table(index='shot_id', columns='cavity', values='PassOrFail', aggfunc='max')
S = L.drop_duplicates('shot_id').set_index('shot_id')
S['LH_fail'], S['RH_fail'] = piv['LH'], piv['RH']
S['pattern'] = np.select([(S['LH_fail'] == 1) & (S['RH_fail'] == 1), S['LH_fail'] == 1, S['RH_fail'] == 1],
                         ['양쪽 불량', 'LH만 불량', 'RH만 불량'], '정상')
display(S.groupby('product')['pattern'].value_counts().unstack(fill_value=0))

def cliffs(a, b):
    u = stats.mannwhitneyu(a, b, alternative='two-sided')
    return 2 * u.statistic / (len(a) * len(b)) - 1, u.pvalue

rows = []
for p in PRODUCTS:
    s = S[S['product'] == p].copy()
    sw = s.copy()
    for v in [T3, T4, 'd']:
        sw[v] = s[v] - s.groupby('ts_date')[v].transform('median')
    ok = s['pattern'] == '정상'
    for pat in ['RH만 불량', 'LH만 불량', '양쪽 불량']:
        f = s['pattern'] == pat
        if f.sum() == 0:
            continue
        r = {'product': p, '패턴': pat, 'n': int(f.sum())}
        for v, lab in [(T3, 'T3'), (T4, 'T4'), ('d', 'T4−T3')]:
            r[f'{lab} δ'], r[f'{lab} p'] = cliffs(s.loc[f, v], s.loc[ok, v])
            r[f'{lab} δ(일자 내)'], _ = cliffs(sw.loc[f, v], sw.loc[ok, v])
        rows.append(r)
cav = pd.DataFrame(rows).set_index(['product', '패턴'])
display(cav.round(2))
print('해석: "T4 = RH"라면 RH만 불량 샷에서 T4 또는 T4−T3의 δ가 일자 내 기준으로도 커야 함 (|δ| ≥ 0.33)')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
for ax, p in zip(axes, PRODUCTS):
    s = S[S['product'] == p].copy()
    s['d_wd'] = s['d'] - s.groupby('ts_date')['d'].transform('median')
    pats = [x for x in ['정상', 'RH만 불량', 'LH만 불량', '양쪽 불량'] if (s['pattern'] == x).any()]
    data = [s.loc[s['pattern'] == x, 'd_wd'] for x in pats]
    ax.boxplot(data, widths=0.5, showfliers=False, medianprops={'color': '#0b0b0b'})
    for i, dvals in enumerate(data, 1):
        ax.scatter(np.random.normal(i, 0.06, len(dvals)), dvals, s=8, alpha=0.5,
                   color=BLUE if pats[i - 1] == '정상' else RED)
    ax.set_xticks(range(1, len(pats) + 1)); ax.set_xticklabels([f'{x}\n(n={len(d)})' for x, d in zip(pats, data)])
    ax.axhline(0, color=GRAY, lw=0.8)
    ax.set_title(f'{p.upper()} · 일자 내 (T4 − T3)', loc='left'); ax.set_ylabel('날짜 중앙값 대비 (℃)')
fig.tight_layout(); fig.savefig(os.path.join(FIG_DIR, 'diff_by_fail_pattern.png'), dpi=150); plt.show()

## ③ unlabeled: 한쪽 캐비티만 기록된 시점

unlabeled에는 같은 `TimeStamp`에 LH만, RH만 있는 경우가 많음. 한쪽만 생산(또는 기록)된 시점에 T3·T4가 어떻게 나오는지 봅니다.
T3 = LH, T4 = RH라면 **LH만 있는 시점엔 T3만, RH만 있는 시점엔 T4만** 값이 있거나 차이 패턴이 뚜렷해야 함.

In [ ]:
for p in PRODUCTS:
    u = U[p]
    kind = u.groupby('TimeStamp')['cavity'].agg(lambda s: ''.join(sorted(s)))
    u['kind'] = u['TimeStamp'].map(kind)
    sh = u.drop_duplicates('TimeStamp')
    t = pd.DataFrame({
        '시점 수': sh.groupby('kind').size(),
        'T3=T4=0 비율': sh.groupby('kind')['zero'].mean().round(3),
        'T3만 0': sh.groupby('kind').apply(lambda g: ((g[T3] == 0) & (g[T4] != 0)).mean()).round(3),
        'T4만 0': sh.groupby('kind').apply(lambda g: ((g[T4] == 0) & (g[T3] != 0)).mean()).round(3),
        'd 중앙값 (0 제외)': sh[~sh['zero']].groupby('kind')['d'].median().round(2),
    })
    print(f'==================== {p.upper()} ====================')
    display(t)

## ④ 정상 기준 대비 T3·T4 변화 방향과 RH 불량

가설: "정상일 때 온도 대비 **T3는 떨어지고 T4는 올라간** 상황에서 RH만 불량이 반복된다" + "더 뜨거워서인지, 더 차가워서인지"

- **정상 기준**: 같은 날, 직전 `BASE_W`개 **정상 샷**의 중앙값 (온도가 하루 안에서도 계단식으로 변해서 날짜 전체 중앙값보다 국소 기준이 적합)
- 각 샷을 T3·T4의 기준 대비 방향(↑ / = / ↓)으로 9개 칸에 나누고, 칸별 RH 불량률 비교
- 불량 유형(`Reason`)도 같이 봄: **미성형 → 차가운 쪽**, **가스 → 뜨거운 쪽(열분해)** 원인과 더 잘 맞음
- ⚠️ 온도 센서 눈금이 0.1℃라 기준 대비 변화도 대부분 ±0.1~0.3℃. 이 크기의 금형 온도 변화가 실제 불량을 만들 만한지는 따로 판단 필요

In [ ]:
BASE_W = 20
S = S.sort_values('TimeStamp')
reason = L[L['PassOrFail'] == 1].groupby('shot_id')['Reason'].agg(lambda s: '/'.join(sorted(set(s.dropna()))))
S['reason'] = reason
print(L[L['PassOrFail'] == 1].groupby(['product', 'cavity'])['Reason'].value_counts().unstack(fill_value=0))

def arrow(x):
    return 'NA' if pd.isna(x) else ('↑' if x > 0 else ('↓' if x < 0 else '='))

QUAD, EVENTS = {}, {}
for p in PRODUCTS:
    s = S[S['product'] == p].copy()
    ok = s['pattern'] == '정상'
    for v, n in [(T3, 'T3'), (T4, 'T4')]:
        base = s[v].where(ok).groupby(s['ts_date']).transform(
            lambda x: x.shift(1).rolling(BASE_W, min_periods=5).median())
        s[f'Δ{n}'] = (s[v] - base).round(2)
    s['방향'] = 'T3' + s['ΔT3'].map(arrow) + ' T4' + s['ΔT4'].map(arrow)
    s['RH불량'] = s['pattern'].isin(['RH만 불량', '양쪽 불량']).astype(int)
    t = s.groupby('방향').agg(샷=('RH불량', 'size'), RH불량=('RH불량', 'sum'))
    t['RH 불량률'] = (t['RH불량'] / t['샷']).round(3)
    t.loc['(전체)'] = [len(s), s['RH불량'].sum(), round(s['RH불량'].mean(), 3)]
    QUAD[p] = t
    EVENTS[p] = s[s['pattern'] != '정상'][['TimeStamp', 'pattern', 'reason', T3, T4, 'ΔT3', 'ΔT4', '방향']]
    print(f'==================== {p.upper()} ====================')
    display(t)
    print('관심 칸 "T3↓ T4↑" 샷 수:', int((s['방향'] == 'T3↓ T4↑').sum()))

In [ ]:
# 불량 샷 하나하나: 언제, 어떤 유형, 기준 대비 T3·T4 변화
for p in PRODUCTS:
    print(f'==================== {p.upper()} 불량 샷 ====================')
    display(EVENTS[p])

In [ ]:
# 시간축 그림: T3·T4 기준 대비 변화 + 불량 샷 위치 (유형별 표시)
for p in PRODUCTS:
    s = S[S['product'] == p].copy()
    ok = s['pattern'] == '정상'
    for v, n in [(T3, 'T3'), (T4, 'T4')]:
        base = s[v].where(ok).groupby(s['ts_date']).transform(
            lambda x: x.shift(1).rolling(BASE_W, min_periods=5).median())
        s[f'Δ{n}'] = s[v] - base
    days = list(s['ts_date'].unique())
    fig, axes = plt.subplots(len(days), 1, figsize=(12, 2.4 * len(days)), squeeze=False)
    for ax, d in zip(axes[:, 0], days):
        g = s[s['ts_date'] == d]
        ax.plot(g['TimeStamp'], g['ΔT3'], color=BLUE, lw=1, label='ΔT3')
        ax.plot(g['TimeStamp'], g['ΔT4'], color=ORANGE, lw=1, label='ΔT4')
        ax.axhline(0, color=GRAY, lw=0.8)
        for pat, mk in [('RH만 불량', 'v'), ('LH만 불량', '^'), ('양쪽 불량', 'D')]:
            f = g[g['pattern'] == pat]
            if len(f):
                ax.scatter(f['TimeStamp'], np.zeros(len(f)), marker=mk, s=40, color=RED, zorder=3, label=pat)
        ax.set_title(f'{p.upper()} · {d}', loc='left', fontsize=9); ax.set_ylabel('기준 대비 ℃', fontsize=8)
    axes[0, 0].legend(frameon=False, fontsize=7, ncol=5, loc='upper right')
    fig.tight_layout(); fig.savefig(os.path.join(FIG_DIR, f'delta_timeline_{p}.png'), dpi=150); plt.show()

## 결론 정리 (실행 후 작성)

| 검증 | CN7 | RG3 | "T3=LH, T4=RH" 지지? |
|---|---|---|---|
| ① T4−T3 부호 | | | |
| ② RH만 불량 샷의 T4·차이 | | | |
| ③ 한쪽만 기록된 시점 | | | |
| ④ T3↓ T4↑ 상황에서 RH 불량 반복? | | | |
| ④ 뜨거운 쪽 / 차가운 쪽 (Reason 포함) | | | |

**다음 할 일**
- [ ] 주최 측 문의: Mold_Temperature 1~n 채널의 실제 센서 위치 (캐비티 LH/RH인지, 냉각수 입·출구인지)
- [ ] 1차 원본의 다른 Mold_Temperature 채널(1, 2, 5, …) 값 확인 — 채널 구성에서 힌트를 얻을 수 있음